# 🚀 ClaimGuard AI V8 (Autonomous Multi-Domain Simulation Engine)
Completely refactored core logical module into a high-performance batch simulator. Removes all audio elements. Native Bengali insight translation supported.

In [ ]:
# ==============================================================================
# CELL 1: Install Dependencies (Audio Removed Completely)
# ==============================================================================

!pip install -q pdf2image gradio faiss-cpu sentence-transformers requests reportlab

!apt-get update -y
!apt-get install -y poppler-utils tesseract-ocr

In [ ]:
# ================================
# CELL 2: 🔧 FIX OLLAMA INSTALL (KAGGLE)
# ================================

import os
import subprocess
import time
import requests

print("🧹 Cleaning old/broken Ollama install...")
!rm -rf /usr/local/lib/ollama
!rm -rf /usr/local/bin/ollama

print("📦 Installing required dependencies...")
!apt-get update -y
!apt-get install -y curl zstd ca-certificates

print("⬇️ Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("🔍 Verifying installation...")
!ls /usr/local/bin/ollama

# ================================
# 🚀 START OLLAMA SERVER
# ================================

print("🚀 Starting Ollama server...")

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["OLLAMA_NUM_PARALLEL"] = "4"
env["OLLAMA_MAX_LOADED_MODELS"] = "2"

ollama_process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(10)

def check_ollama():
    try:
        r = requests.get("http://localhost:11434/api/tags")
        return r.status_code == 200
    except:
        return False

if check_ollama():
    print("✅ Ollama is RUNNING!")
else:
    print("❌ Ollama failed to start")

print("📥 Pulling llama3 model (first time takes time)...")
!ollama pull llama3:8b
print("✅ Setup Complete!")

In [ ]:
# ==============================================================================
# CELL 3: Imports & Configuration (Removed all audio APIs)
# ==============================================================================

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import io
import re
import json
import time
import base64
import requests
import faiss
import numpy as np
import torch
from pdf2image import convert_from_path
from sentence_transformers import SentenceTransformer
import gradio as gr

OCR_SPACE_API_KEY = "K87093604688957"
OLLAMA_API_URL = "http://localhost:11434/api/generate"
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'

print("Loading Embedding Model (CPU safe mode)...")
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device="cpu")
print("✅ Embedding model loaded successfully on CPU")

vector_db = None
clause_store = []
pdf_cache = {}

In [ ]:
# CELL 4: 1. OCR MODULE (OCR.space Integration)
def extract_text_from_pdf(pdf_path):
    print("Converting PDF to images for OCR...")
    images = convert_from_path(pdf_path, dpi=100) 
    full_text = ""
    
    for i, img in enumerate(images):
        print(f"Processing Page {i+1} via OCR.space API...")
        img_byte_arr = io.BytesIO()
        img.save(img_byte_arr, format='JPEG', quality=40) 
        base64_encoded = base64.b64encode(img_byte_arr.getvalue()).decode('utf-8')
        
        payload = {
            'apikey': OCR_SPACE_API_KEY,
            'base64Image': f"data:image/jpeg;base64,{base64_encoded}",
            'language': 'eng',
            'isOverlayRequired': False
        }
        
        try:
            response = requests.post('https://api.ocr.space/parse/image', data=payload)
            result = response.json()
            if result.get('IsErroredOnProcessing') == False:
                parsed_text = result['ParsedResults'][0]['ParsedText']
                full_text += parsed_text + "\n\n"
            else:
                print(f"API Error on Page {i+1}: {result.get('ErrorMessage')}")
        except Exception as e:
            print(f"Request failed for Page {i+1}: {e}")
            
        time.sleep(1) 
        
    return full_text

In [ ]:
# CELL 5: 2. NLP TEXT PROCESSING MODULE
def process_text(raw_text):
    print("Cleaning and chunking text...")
    cleaned = re.sub(r'\s+', ' ', raw_text).strip()
    words = cleaned.split()
    chunk_size = 300
    overlap = 50
    chunks = [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]

    categories = {
        "waiting_period": ["waiting", "days from inception", "delay"],
        "exclusion": ["excluded", "not covered", "exclusion", "except"],
        "co_payment": ["co-pay", "copayment", "co payment", "%"],
        "room_rent": ["room rent", "bed charges", "icu"],
        "sub_limit": ["sub-limit", "capped at", "maximum limit"],
        "network_rule": ["network", "empanelled", "hospital"]
    }
    
    structured_clauses = []
    for chunk in chunks:
        cls_tags = []
        lower_chunk = chunk.lower()
        for cat, keywords in categories.items():
            if any(k in lower_chunk for k in keywords):
                cls_tags.append(cat)
        structured_clauses.append({
            "text": chunk,
            "categories": cls_tags if cls_tags else ["general"]
        })
        
    print(f"Extracted {len(structured_clauses)} structural clause chunks.")
    return structured_clauses

In [ ]:
# CELL 6: 3. EMBEDDING + VECTOR DB (FAISS)
def build_vector_db(structured_clauses):
    global vector_db, clause_store
    clause_store = structured_clauses
    texts = [c['text'] for c in structured_clauses]
    
    print("Generating FAISS Embeddings...")
    embeddings = embedder.encode(texts).astype('float32')
    
    dimension = embeddings.shape[1]
    vector_db = faiss.IndexFlatL2(dimension)
    vector_db.add(embeddings)
    print(f"Vector DB initialized with {vector_db.ntotal} clause embeddings.")

def get_relevant_clauses(query, top_k=5):
    if not vector_db: return []
    query_vector = embedder.encode([query]).astype('float32')
    distances, indices = vector_db.search(query_vector, top_k)
    return [clause_store[idx] for idx in indices[0] if idx < len(clause_store)]

In [ ]:
# CELL 7: 4. SCENARIO PARSER (Ollama JSON Parsing with bump timeout)
def call_ollama(prompt, system_prompt, json_format=True, timeout=1200):
    payload = {
        "model": "llama3:8b",
        "prompt": prompt,
        "system": system_prompt,
        "stream": False
    }
    if json_format:
        payload["format"] = "json"
        
    try:
        response = requests.post(OLLAMA_API_URL, json=payload, timeout=timeout)
        response.raise_for_status()
        return response.json().get('response', '{}' if json_format else '')
    except Exception as e:
        print(f"Ollama inference error: {e}")
        return "{}" if json_format else ""

def parse_scenario(user_query):
    sys_prompt = "Convert user healthcare scenario to structured JSON with exactly keys: treatment, hospital, urgency, input_raw, estimated_cost."
    response = call_ollama(user_query, system_prompt=sys_prompt, timeout=120)
    try:
        return json.loads(response)
    except:
        return {"input_raw": user_query, "hospital": "Unknown", "treatment": "Unknown"}

In [ ]:
# CELL 8: AUTONOMOUS MULTI-DOMAIN SIMULATION ENGINE
def multi_domain_simulation_engine(user_query, context_clauses):
    print("🧠 Engaging Autonomous Multi-Domain Simulation Engine (Target: 100 Scenarios) ...")
    context_text = "\n".join([f"- {c['text']}" for c in context_clauses])
    
    sys_prompt = """You are an Autonomous Multi-Domain Simulation and Reasoning Engine optimized for HIGH PERFORMANCE and PARALLEL EXECUTION.

Your goal is to fully automate understanding, simulation, evaluation, and insight generation for ANY input.

You DO NOT ask the user questions.
You DO NOT wait for user interaction.
You EXECUTE everything automatically in ONE FLOW.

================================
PHASE 1: CONTEXT UNDERSTANDING
================================
- Analyze the input deeply.
- Identify: domain, entities, rules, constraints, decision logic, risks

================================
PHASE 2: HIGH-QUALITY SIMULATION GENERATION
================================
- Generate EXACTLY 100 UNIQUE scenarios.
- Each scenario MUST be: non-redundant, diverse across conditions, include normal + edge + adversarial cases
Each scenario:
{ "scenario_id": number, "context": "...", "input_parameters": {...}, "environment": "...", "risk_level": "low/medium/high", "edge_case": true/false, "expected_goal": "..." }

================================
PHASE 3: COLLECTIVE BATCH EVALUATION
================================
For EACH scenario evaluate outcome conceptualized in parallel blocks:
{ "scenario_id": number, "outcome": "...", "status": "success/failure/partial", "reason": "...", "rule_applied": "...", "confidence": number }

================================
PHASE 4: SYSTEM INTELLIGENCE
================================
Analyze all results and detect: system weaknesses, failure patterns, high-risk scenarios, rule gaps, optimization opportunities

================================
PHASE 5: FINAL OUTPUT (STRICT JSON ONLY)
================================
Return ONE final JSON:
{
  "context_analysis": {
    "domain": "...", "entities": [], "rules": [], "constraints": [], "decision_criteria": [], "risk_factors": []
  },
  "total_simulations": 100,
  "simulation_results": [
     {
        "scenario": { "scenario_id": 1, "context": "..." },
        "evaluation": { "outcome": "...", "status": "..." }
     }
  ],
  "insights": {
    "system_weaknesses": [], "failure_patterns": [], "high_risk_scenarios": [], "optimization_suggestions": []
  }
}

================================
PERFORMANCE RULES (VERY IMPORTANT)
================================
- EXACTLY 100 scenarios (not more, not less)
- Minimize token usage, keep nested text extremely concise
- Avoid redundancy
- No repeated explanations
- No external APIs
- No audio generation
- Output valid JSON only
"""

    prompt = f"""
INPUT CONTEXT:
- Concept/Query: {user_query}
- Extracted Policy Constraints:
{context_text}

EXECUTE PHASE 1 TO 5 NOW. GENERATE 100 SCENARIOS AND RETURN THE FINAL JSON BLOCK.
"""
    
    try:
        resp = call_ollama(prompt, system_prompt=sys_prompt, json_format=True, timeout=2400)
        output = json.loads(resp)
        print(f"✅ Generated Simulation Batch with {len(output.get('simulation_results', []))} scenarios.")
        return output
    except Exception as e:
        print(f"⚠️ Simulation Engine Overloaded or Failed: {e}")
        return {"error": str(e)}


In [ ]:
# CELL 9: 🌍 INSIGHT TRANSLATION MODULE (Bengali only as requested)
def translate_insights_to_bengali(simulation_json):
    print("🌍 Translating final insights to simple Bengali...")
    insights = simulation_json.get("insights", {})
    if not insights:
        return "কোনো ইনসাইট পাওয়া যায়নি। (No insights found.)"
        
    insights_text = json.dumps(insights, indent=2)
    sys_prompt = "You are a specialized translator. Read the provided system insights and summarize/translate them strictly into very simple Bengali language text suitable for older generations. Output ONLY Bengali language text."
    
    prompt = f"Translate these system insights into simple Bengali:\n\n{insights_text}"
    
    bengali_translation = call_ollama(prompt, system_prompt=sys_prompt, json_format=False, timeout=300).strip()
    
    if not bengali_translation:
        return "অনুবাদ এই মুহূর্তে উপলব্ধ নয়।"
        
    return bengali_translation.encode("utf-8").decode("utf-8")

In [ ]:
# CELL 10: FULL SIMULATION PIPELINE INTEGRATION
def run_simulation_pipeline(pdf_file, user_query):
    pdf_path = pdf_file.name if hasattr(pdf_file, 'name') else str(pdf_file)
    
    if pdf_path not in pdf_cache:
        raw_text = extract_text_from_pdf(pdf_path)
        clauses = process_text(raw_text)
        build_vector_db(clauses)
        pdf_cache[pdf_path] = True
    else:
        print("=> Using cached Embeddings for this PDF")

    scenario = parse_scenario(user_query)
    context_clauses = get_relevant_clauses(user_query, top_k=7) # Broader context for 100 scenarios
    
    # Massive 100-scenario generation engine
    simulation_output_json = multi_domain_simulation_engine(user_query, context_clauses)
    
    # Optional Bengali translation for the insights block
    bengali_insights = translate_insights_to_bengali(simulation_output_json)
    
    return simulation_output_json, bengali_insights

In [ ]:
# CELL 11: AUTOMATED TEST CASE & UI
import warnings
warnings.filterwarnings('ignore')

def create_dummy_pdf(filename="sample_policy.pdf"):
    try:
        from reportlab.pdfgen import canvas
        c = canvas.Canvas(filename)
        c.drawString(100, 750, "ClaimGuard Sample Health Insurance Policy")
        c.drawString(100, 730, "1. This policy covers all emergency treatments and accident surgeries.")
        c.drawString(100, 710, "2. A 10% co-payment (co-pay) is mandatory for all surgical claims.")
        c.drawString(100, 690, "3. Sub-limit: Room rent is capped at ₹5,000 per day.")
        c.drawString(100, 670, "4. Waiting period: 30 days initial waiting period from inception.")
        c.drawString(100, 650, "5. Exclusions: Cosmetic surgery is strictly excluded.")
        c.drawString(100, 630, "6. Empanelled network hospital Apollo is fully covered.")
        c.save()
    except Exception as e:
        pass

create_dummy_pdf()

def gradio_wrapper(pdf, query):
    if not pdf or not query: 
        return {"error": "Please upload a PDF and specify an input to simulate."}, ""
    try:
        return run_simulation_pipeline(pdf, query)
    except Exception as e:
        return {"error": f"Pipeline Error: {str(e)}"}, ""

interface = gr.Interface(
    fn=gradio_wrapper,
    inputs=[
        gr.File(label="📄 Context Input (PDF Supported)"),
        gr.Textbox(label="🧠 Simulation Parameter Setup", placeholder="What should we simulate 100 times?")
    ],
    outputs=[
        gr.JSON(label="🖥️ High Performance Simulation Results (Raw Data Block)"),
        gr.Markdown(label="🇧🇩 সহজ বাংলা অনুবাদ (Insights Output)")
    ],
    title="ClaimGuard AI 🤖 - Autonomous Multi-Domain Simulation Engine",
    description="Upload context data to automatically execute 100 parallel batch evaluations, map domain constraints, and extract adversarial system insights."
)

try:
    _ = interface.launch(share=True, inline=False, quiet=True)
    print("✅ Massive Simulation Engine Web UI Deployed! Use the public *.gradio.live link above.")
except Exception as e:
    print(f"Gradio Launch Error: {e}")